# WSJ 2027 - Gruppindelning Rundresa

Assign confirmed rundresa deltagare into groups of exactly 36.

## Rules
1. **Exactly 36 per group** (remainder group allowed)
2. **Geographic proximity** — participants should live close to each other (Hilbert curve sort, friend-cluster-aware)
3. **Friend wish** — at least ONE of friend_1/friend_2 in same group (soft goal)
4. **Max 8 from same kår** per group (hard constraint)
5. **Diversity** — age (14-17) and sex should be as evenly spread as possible


In [1]:
TRAVEL = 'rundresa'        # only line that differs between rundresa.ipynb and direktresa.ipynb
QUALITY = 'medium'         # 'medium' (~1-3 min) or 'slow' (~10-30 min, +5-10% friends)
GROUP_SIZE = 36
OUTPUT_DIR = '/config/notebooks/wsj27/output'


In [2]:
import sys; sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u

raw = u.fetch_participants()
df_all, _ = u.build_participant_dataframe(raw)
df = df_all[df_all['travel'] == TRAVEL].copy().reset_index(drop=True)
u.assign_coordinates(df)
df = u.add_hilbert_index(df)
u.resolve_friend_wishes(df, df_all)
friend_wishes = u.build_friend_graph(df)
u.print_intake_summary(df, GROUP_SIZE)


Fetched 2478 participants
Confirmed: 2397, Unconfirmed: 3, Cancelled: 78
Total confirmed participants: 2397
Skipped: 3 unconfirmed, 0 wrong age/no DOB

By category:
category
deltagare    1930
ist           445
cmt            22

By travel type:
travel
rundresa      1506
direktresa     569
egen_resa      300
other           22

Friend wishes:
  With friend 1 member no: 1355
  With friend 2 member no: 803
  With friend 1 name (text): 118
  With friend 2 name (text): 80
With coordinates: 1504
Without coordinates (Sweden centroid): 2
=== Text Friend Name Matching ===
Text-only wishes: 111
Matched & applied: 78
Generic wishes (not a person): 4
Unresolved (friend not in project): 29

Matched:
  Adelia Vallin -> Leo Norstedt (S:ta Maria Scoutkår) [rundresa] via fuzzy(0.96)
  Albin Åkesson -> Johannes Hellsten (Borlänge Scoutkår) [direktresa] via fuzzy(0.76)
  Albin Åkesson -> Malte Ekström (Lomma scoutkår) [rundresa] via fuzzy(0.84)
  Alex Liljerot Priftis -> Vera Tollwé (Årsta Scoutkår) [run

In [ ]:
df = u.assign_groups(df, GROUP_SIZE, friend_wishes, quality=QUALITY)
group_of = df['group'].values
total_groups = df['group'].nunique()


Participants: 1506
Groups: 41 x 36 + 1 x 30 = 42 total

=== Phase 1: Geographic sort + cut ===
  Friend satisfaction: 943/943
  Kar violations: 154
  Avg geo spread: 2.3298

=== Phase 2: Fix friend wishes (criticality-ordered, iterate) ===
  No unsatisfied wishes; converged after 0 pass(es)
  Total swaps: 0
  Friend satisfaction: 943/943
  Kar violations: 154
  Avg geo spread: 2.3298

=== Phase 2.5: 3-way rotations ===
  Rotations: 0
  Friend satisfaction: 943/943
  Kar violations: 154

=== Phase 3: Fix kar violations (friend-aware) ===
  Swaps: 146
  Kar violations: 0
  Friend satisfaction: 843/943
  Avg geo spread: 2.3749

=== Phase 2b: Re-fix friends after kar fix ===
  Swaps: 66
  Friend satisfaction: 931/943
  Kar violations: 0
  Avg geo spread: 2.3918

=== Phase 4: Weighted SA (friend-positive) ===


In [ ]:
u.print_group_metrics(df, group_of, total_groups)
csv_path, json_path = u.export_results(df, group_of, total_groups, OUTPUT_DIR, prefix=f'wsj27_{TRAVEL}')
map_path = f'{OUTPUT_DIR}/wsj_{TRAVEL}_karta.html'
u.generate_group_map_html(df, total_groups, map_path,
                          title=f'WSJ 2027 {TRAVEL.title()}',
                          friend_wishes=friend_wishes, show_group_arcs=True)
print(f'CSV:  {csv_path}\nJSON: {json_path}\nMap:  {map_path}')
